In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.signal import periodogram
from prophet import Prophet
import xgboost as xgb
import lightgbm as lgb
import warnings
from itertools import product

In [ ]:
df_eda=pd.read_csv("agric.csv");df_eda

In [ ]:
df_eda["Arrival_Date"] = pd.to_datetime(df_eda["Arrival_Date"])
df_eda = df_eda.sort_values("Arrival_Date")
df_eda.reset_index(drop=True, inplace=True)
df_eda

In [ ]:
print("🔹 Dataset Info:")
print(df_eda.info())
print("\n🔹 Missing Values:")
print(df_eda.isnull().sum())
print("\n🔹 Summary Stats:")
print(df_eda.describe())

In [ ]:
weather_cols = ['Avg_Temp_C', 'Min_Temp_C', 'Max_Temp_C', 'Rainfall_mm']

df_eda = df_eda.sort_values(by=['Market', 'Arrival_Date'])

# Convert 'Arrival_Date' to datetime
df_eda['Arrival_Date'] = pd.to_datetime(df_eda['Arrival_Date'])

# Market-wise forward and backward fill using transform
for col in weather_cols:
    df_eda[col] = df_eda.groupby('Market')[col].transform(lambda x: x.ffill().bfill())

# Fill remaining missing values using market-wise mean
for col in weather_cols:
    df_eda[col] = df_eda.groupby('Market')[col].transform(lambda x: x.fillna(x.mean()))

# Final backup using market-month average (for seasonal effect)
df_eda['Month'] = df_eda['Arrival_Date'].dt.month
for col in weather_cols:
    df_eda[col] = df_eda.groupby(['Market', 'Month'])[col].transform(lambda x: x.fillna(x.mean()))
df_eda

In [ ]:
print(df_eda.isnull().sum())


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown

# 1. Create the interactive widget first
#    Using sorted() makes the dropdown list alphabetical and easier to use.
commodity_dropdown = Dropdown(
    options=sorted(df_eda["Commodity"].unique().tolist()),
    description="Commodity:",
    value=sorted(df_eda["Commodity"].unique().tolist())[6],
    style={'description_width': 'initial'},
    layout={'width': '300px'}
)

# 2. Define the plotting function (no changes needed here)
def plot_price_trend(commodity):
    temp = df_eda[df_eda["Commodity"] == commodity]

    # Check if there's any data to plot
    if temp.empty:
        print(f"No data available for {commodity}.")
        return

    plt.figure(figsize=(12, 6))
    sns.lineplot(x="Arrival_Date", y="Modal_Price", data=temp, alpha=0.7)
    plt.title(f"Price Trend for {commodity}")
    plt.xlabel("Date")
    plt.ylabel("Modal Price (Rs/Quintal)")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 3. Call interact and pass the widget you already created
interact(plot_price_trend, commodity=commodity_dropdown);

In [ ]:
commodity_dropdown = Dropdown(
    options=sorted(df_eda["Commodity"].unique().tolist()),
    description="Commodity:",
    value=sorted(df_eda["Commodity"].unique().tolist())[1],
    style={'description_width': 'initial'},
    layout={'width': '300px'}
)
def plot_rainfall_vs_price(commodity):
    temp = df_eda[df_eda["Commodity"] == commodity]
    plt.figure(figsize=(8,6))
    sns.scatterplot(x="Rainfall_mm", y="Modal_Price", data=temp, alpha=0.5)
    plt.title(f"Rainfall vs Price - {commodity}")
    plt.xlabel("Rainfall (mm)")
    plt.ylabel("Modal Price")
    plt.show()

interact(plot_rainfall_vs_price, commodity=Dropdown(options=df_eda["Commodity"].unique(), description="Commodity:"))


In [ ]:

# 1. Create the interactive widget first
#    Using sorted() makes the dropdown list alphabetical and easier to use.
commodity_dropdown = Dropdown(
    options=sorted(df_eda["Commodity"].unique().tolist()),
    description="Commodity:",
    value=sorted(df_eda["Commodity"].unique().tolist())[4],
    style={'description_width': 'initial'},
    layout={'width': '300px'}
)
def plot_temp_vs_price(commodity):
    temp = df_eda[df_eda["Commodity"] == commodity]
    plt.figure(figsize=(8,6))
    sns.scatterplot(x="Avg_Temp_C", y="Modal_Price", data=temp, alpha=0.5)
    plt.title(f"Temperature vs Price - {commodity}")
    plt.xlabel("Avg Temperature (°C)")
    plt.ylabel("Modal Price")
    plt.show()
interact(plot_temp_vs_price, commodity=Dropdown(options=df_eda["Commodity"].unique(), description="Commodity:"))


In [ ]:
def plot_market_distribution(commodity):
    temp = df_eda[df_eda["Commodity"] == commodity]
    plt.figure(figsize=(12,6))
    sns.boxplot(x="Market", y="Modal_Price", data=temp)
    plt.xticks(rotation=90)
    plt.title(f"Price Distribution by Market - {commodity}")
    plt.show()

interact(plot_market_distribution, commodity=Dropdown(options=df_eda["Commodity"].unique(), description="Commodity:"))

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(x="Market", y="Modal_Price", data=df)
plt.xticks(rotation=90)
plt.title("Modal Price Distribution per Market")
plt.show()

In [ ]:
df_eda['Modal_Price_Smooth'] = df_eda.groupby('Market')['Modal_Price'].transform(lambda x: x.rolling(7, min_periods=1).mean())

top_markets = df_eda['Market'].value_counts().nlargest(5).index.tolist()

df_top = df_eda[df_eda['Market'].isin(top_markets)]

plt.figure(figsize=(14, 6))
sns.lineplot(
    data=df_top,
    x='Arrival_Date',
    y='Modal_Price_Smooth',
    hue='Market',
    alpha=0.8
)
plt.title('Smoothed Modal Price Trends (Top 5 Markets)')

plt.xlabel('Date')
plt.ylabel('7-Day Avg Modal Price (Rs/Quintal)')
plt.xticks(rotation=45)
plt.legend(title='Market', loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True)
plt.tight_layout()
plt.show()


#Anomaly Detection

In [ ]:
def run_anomaly_detection(commodity):
    series = df_eda[df_eda['Commodity'] == commodity][["Arrival_Date", "Modal_Price", "Rainfall_mm", "Avg_Temp_C"]]
    series = series.groupby("Arrival_Date").mean().reset_index()
    series = series.set_index("Arrival_Date").sort_index()
    series = series.asfreq("D").fillna(method="ffill")

    print(f" Running anomaly detection for: {commodity}")

    data_for_anomaly = series[["Modal_Price", "Rainfall_mm", "Avg_Temp_C"]].values

    iso = IsolationForest(contamination=0.05, random_state=42)
    labels = iso.fit_predict(data_for_anomaly)

    # Identify anomalies
    anomalies = series[labels == -1]

    print(f"\n Detected {len(anomalies)} anomalies.")

    plt.figure(figsize=(12, 6))
    plt.plot(series.index, series["Modal_Price"], label='Modal Price', alpha=0.8)
    plt.scatter(anomalies.index, anomalies["Modal_Price"], color='red', label='Anomalies', s=50)
    plt.title(f'Anomaly Detection for {commodity}')
    plt.xlabel('Date')
    plt.ylabel('Scaled Modal Price')
    plt.legend()
    plt.grid(True)
    plt.show()

commodity_dropdown_anomaly = Dropdown(
    options=df_eda['Commodity'].unique().tolist(),
    description='Select Commodity for Anomaly Detection:',
    value=df_eda['Commodity'].unique()[0],  # default first commodity
    style={'description_width': 'initial'},
    layout={'width': '400px'}
)

interact(run_anomaly_detection, commodity=commodity_dropdown_anomaly);

In [ ]:
%matplotlib inline

In [ ]:
df=pd.read_csv("agric.csv");df

In [ ]:
weather_cols = ['Avg_Temp_C', 'Min_Temp_C', 'Max_Temp_C', 'Rainfall_mm']

df = df.sort_values(by=['Market', 'Arrival_Date'])

# Convert 'Arrival_Date' to datetime
df['Arrival_Date'] = pd.to_datetime(df['Arrival_Date'])

# Market-wise forward and backward fill using transform
for col in weather_cols:
    df[col] = df_eda.groupby('Market')[col].transform(lambda x: x.ffill().bfill())

# Fill remaining missing values using market-wise mean
for col in weather_cols:
    df[col] = df.groupby('Market')[col].transform(lambda x: x.fillna(x.mean()))

# Final backup using market-month average (for seasonal effect)
df['Month'] = df['Arrival_Date'].dt.month
for col in weather_cols:
    df[col] = df.groupby(['Market', 'Month'])[col].transform(lambda x: x.fillna(x.mean()))
df

In [ ]:
#remove the month column
df.drop(columns=['Month'], inplace=True);df

In [ ]:
df.isnull().sum()

In [ ]:
# Mount Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

# Imports
import os, time, math, warnings, uuid, joblib, pickle, multiprocessing as mp, glob
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")
%matplotlib inline

# Optional libs
try:
    import pmdarima as pm
    HAS_PMD = True
except Exception:
    HAS_PMD = False

try:
    from prophet import Prophet
    HAS_PROPHET = True
except Exception:
    HAS_PROPHET = False

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False
from sklearn.svm import SVR
try:
    import optuna
    HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False

# --------------------------
# USER CONFIG (edit)
# --------------------------
# For sample quick test set to sample path; for full run point to full CSV in Drive
CSV_PATH = "/content/drive/MyDrive/agric.csv"   # <- set to sample first, then full file
OUTPUT_DIR = "/content/drive/MyDrive/agric_forecast_results_run"
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
PARTIAL_DIR = os.path.join(OUTPUT_DIR, "partials")
for d in [OUTPUT_DIR, PLOT_DIR, MODEL_DIR, PARTIAL_DIR]:
    os.makedirs(d, exist_ok=True)

MASTER_FINAL = os.path.join(OUTPUT_DIR, "master_summary_cv.csv")
BEST_FINAL = os.path.join(OUTPUT_DIR, "best_model_summary.csv")
FORECAST_FINAL = os.path.join(OUTPUT_DIR, "future_forecast.csv")
# per-run forecast checkpoint (resumable)
forecast_ckpt = os.path.join(PARTIAL_DIR, "future_forecast_partial.csv")

# Runtime-friendly defaults (change if you want)
MIN_OBS = 365
LAGS = [1,2,3,7,14,21,28]
N_CV_SPLITS = 2             # keep 2 folds for rolling CV
ETS_SEASONAL_PERIODS = 7
OPTUNA_TRIALS = 2           # Optuna only for XGBoost; set 0 to disable
HORIZON = 30
RANDOM_STATE = 42
N_WORKERS = 2               # parallel workers in Pool
AUTO_MERGE_EVERY = 10       # auto-merge partials every N completed pairs (0 to disable)
BOOTSTRAP_N = 200
ALPHA = 0.05

np.random.seed(RANDOM_STATE)

In [ ]:
def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def make_lag_features(series, exog_df, lags):
    X = pd.DataFrame(index=series.index)
    for lag in lags:
        X[f"lag_{lag}"] = series.shift(lag)
    X["roll_mean_7"] = series.shift(1).rolling(7).mean()
    if exog_df is not None:
        X = X.join(exog_df)
    X = X.dropna()
    y_aligned = series.loc[X.index]
    return X, y_aligned

def ensure_stationary(y):
    try:
        result = adfuller(y.dropna())
        pval = result[1]
        if pval > 0.05:
            return y.diff().dropna()
        return y
    except Exception:
        return y



In [ ]:
def sanitize_name(s):
    return "".join(c if c.isalnum() or c in (" ", "_", "-") else "_" for c in str(s)).replace(" ", "_")

def rmse(y_true, y_pred): return math.sqrt(mean_squared_error(y_true, y_pred))
def mae(y_true, y_pred): return mean_absolute_error(y_true, y_pred)
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def adf_pvalue(series):
    try:
        return adfuller(series.dropna())[1]
    except Exception:
        return np.nan

def ensure_stationary(y):
    try:
        pval = adfuller(y.dropna())[1]
        if pval > 0.05:
            return y.diff().dropna()
        return y
    except Exception:
        return y

In [ ]:
def clean_full_df(df):
    df = df.copy()
    df = df.drop_duplicates()
    df['Modal_Price'] = pd.to_numeric(df['Modal_Price'], errors='coerce')
    if 'Rainfall_mm' in df.columns:
        df['Rainfall_mm'] = pd.to_numeric(df['Rainfall_mm'], errors='coerce')
    if 'Avg_Temp_C' in df.columns:
        df['Avg_Temp_C'] = pd.to_numeric(df['Avg_Temp_C'], errors='coerce')
    df = df.dropna(subset=['Arrival_Date','Market','Commodity','Modal_Price'])
    df['Arrival_Date'] = pd.to_datetime(df['Arrival_Date'], errors='coerce')
    df = df.dropna(subset=['Arrival_Date'])
    low_q = df['Modal_Price'].quantile(0.005)
    high_q = df['Modal_Price'].quantile(0.995)
    df['Modal_Price'] = df['Modal_Price'].clip(lower=low_q, upper=high_q)
    return df

In [ ]:
def make_lag_features(series, exog_df, lags):
    X = pd.DataFrame(index=series.index)
    for lag in lags:
        X[f"lag_{lag}"] = series.shift(lag)
    X["roll_mean_7"] = series.shift(1).rolling(7).mean()
    if exog_df is not None and not exog_df.empty:
        X = X.join(exog_df)
    X = X.dropna()
    y_aligned = series.loc[X.index]
    return X, y_aligned

In [ ]:
def iterative_forecast_ml(model, y_train, exog_train, exog_future, horizon, lags, feature_cols, scaler=None):
    history = y_train.copy().astype(float)
    full_exog = None
    if exog_train is not None and exog_future is not None:
        full_exog = pd.concat([exog_train, exog_future])
    elif exog_train is not None:
        full_exog = exog_train
    elif exog_future is not None:
        full_exog = exog_future
    preds = []
    for h in range(1, horizon+1):
        idx = history.index[-1] + pd.Timedelta(days=1)
        feat = {}
        for lag in lags:
            look_idx = idx - pd.Timedelta(days=lag)
            if look_idx in history.index:
                feat[f"lag_{lag}"] = history.loc[look_idx]
            else:
                feat[f"lag_{lag}"] = history.iloc[-1]
        feat['roll_mean_7'] = history.iloc[-7:].mean() if len(history) >= 7 else history.mean()
        if full_exog is not None:
            for c in full_exog.columns:
                if idx in full_exog.index:
                    feat[c] = full_exog.loc[idx, c]
                else:
                    feat[c] = full_exog.iloc[-1][c] if len(full_exog) > 0 else np.nan
        for c in feature_cols:
            if c not in feat:
                feat[c] = np.nan
        feat_df = pd.DataFrame([feat], index=[idx])[feature_cols]
        feat_df = feat_df.fillna(method='ffill').fillna(method='bfill').fillna(0.0)
        X_pred = scaler.transform(feat_df) if scaler is not None else feat_df
        p = model.predict(X_pred)
        pval = float(p[0]) if hasattr(p, '__len__') else float(p)
        preds.append(pval)
        history.loc[idx] = pval
    idxs = pd.date_range(start=y_train.index[-1] + pd.Timedelta(days=1), periods=horizon, freq='D')
    return pd.Series(preds, index=idxs)

In [ ]:
def bootstrap_intervals(pred_series, residuals, horizon=30, n_boot=200, alpha=0.05):
    pred_vals = np.array(pred_series)
    boot = []
    for _ in range(n_boot):
        noise = np.random.choice(residuals, size=horizon, replace=True)
        boot.append(pred_vals + noise)
    boot = np.array(boot)
    lower = np.percentile(boot, 100*alpha/2, axis=0)
    upper = np.percentile(boot, 100*(1-alpha/2), axis=0)
    lower = np.maximum(lower, 0.0)
    return lower, upper

In [ ]:
def select_best_metric(scores):
    errs = {}
    for k in ['rmse','mae','mape']:
        v = scores.get(k, np.nan)
        if not np.isnan(v):
            errs[k.upper()] = v
    if not errs:
        return None
    best_err = min(errs, key=errs.get)
    vals = list(errs.values())
    if max(vals) - min(vals) < 0.05 * min(vals):
        return "Balanced"
    return best_err

In [ ]:
def optuna_tune(model_name, X, y, n_trials=OPTUNA_TRIALS, random_state=RANDOM_STATE):
    if not HAS_OPTUNA or n_trials <= 0 or len(X) < 10:
        return None
    split = int(len(X)*0.8)
    X_tr, X_val = X.iloc[:split], X.iloc[split:]
    y_tr, y_val = y.iloc[:split], y.iloc[split:]

    def objective(trial):
        if model_name == "XGBoost":
            params = {
                "max_depth": trial.suggest_int("max_depth", 2, 8),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                "n_estimators": trial.suggest_int("n_estimators", 50, 300),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "random_state": random_state
            }
            m = XGBRegressor(**params, verbosity=0)
            m.fit(X_tr, y_tr)
            preds = m.predict(X_val)
            return rmse(y_val, preds)
        elif model_name == "RandomForest":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 50, 300),
                "max_depth": trial.suggest_int("max_depth", 3, 20),
                "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
                "random_state": random_state
            }
            m = RandomForestRegressor(**params)
            m.fit(X_tr, y_tr)
            preds = m.predict(X_val)
            return rmse(y_val, preds)
        elif model_name == "SVR":
            params = {
                "C": trial.suggest_float("C", 0.1, 50.0, log=True),
                "epsilon": trial.suggest_float("epsilon", 0.001, 1.0, log=True),
                "gamma": trial.suggest_categorical("gamma", ["scale", "auto"])
            }
            scaler = StandardScaler().fit(X_tr)
            m = SVR(**params)
            m.fit(scaler.transform(X_tr), y_tr)
            preds = m.predict(scaler.transform(X_val))
            return rmse(y_val, preds)
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

In [ ]:
def save_artifact(model_obj, model_name, market, commodity, scaler=None, feature_cols=None, meta=None):
    fname = f"model_{sanitize_name(market)}_{sanitize_name(commodity)}_{model_name}.joblib"
    path = os.path.join(MODEL_DIR, fname)
    artifact = {"model": model_obj, "scaler": scaler, "feature_cols": feature_cols, "meta": meta or {}}
    joblib.dump(artifact, path)
    return path


In [ ]:
def run_pair_worker(args):
    market, commodity, df_full = args
    uid = f"{os.getpid()}_{uuid.uuid4().hex[:8]}"
    try:
        sub_raw = df_full[(df_full['Market'] == market) & (df_full['Commodity'] == commodity)].copy()
        if len(sub_raw) < MIN_OBS:
            return {"status":"skip","market":market,"commodity":commodity,"reason":"too_few_obs","count":len(sub_raw)}

        sub_raw['Arrival_Date'] = pd.to_datetime(sub_raw['Arrival_Date'])
        sub_agg = sub_raw.groupby('Arrival_Date').agg({'Modal_Price':'median','Rainfall_mm':'mean','Avg_Temp_C':'mean'}).asfreq('D').interpolate()
        y = sub_agg['Modal_Price'].dropna()
        if y.empty:
            return {"status":"skip","market":market,"commodity":commodity,"reason":"empty_after_agg"}

        exog = sub_agg[['Rainfall_mm','Avg_Temp_C']].loc[y.index] if 'Rainfall_mm' in sub_agg.columns else None

        # STL plot (optional)
        try:
            stl = STL(y, period=7).fit()
            fig = stl.plot()
            plt.suptitle(f"STL - {market} - {commodity}")
            pfn = os.path.join(PLOT_DIR, f"STL_{sanitize_name(market)}_{sanitize_name(commodity)}.png")
            plt.tight_layout(); plt.savefig(pfn, bbox_inches='tight'); plt.close()
        except Exception:
            pass

        # Cross-validation
        tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS, test_size=30)
        model_scores = {m: {"rmse": [], "mae": [], "mape": []} for m in ['AutoARIMA','ETS','Prophet','XGBoost','RandomForest','SVR']}
        last_preds = {}

        for train_idx, test_idx in tscv.split(y):
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            exog_train = exog.iloc[train_idx] if exog is not None else None
            exog_test  = exog.iloc[test_idx]  if exog is not None else None

            # ---------- AutoARIMA ----------
            if HAS_PMD:
                try:
                    m_ar = pm.auto_arima(y_train, exogenous=exog_train, seasonal=True, m=7,
                                         stepwise=True, suppress_warnings=True, error_action='ignore',
                                         max_p=2, max_q=2)
                    fc = m_ar.predict(n_periods=len(y_test), exogenous=exog_test)
                    model_scores['AutoARIMA']['rmse'].append(rmse(y_test, fc))
                    model_scores['AutoARIMA']['mae'].append(mae(y_test, fc))
                    model_scores['AutoARIMA']['mape'].append(mape(y_test, fc))
                    last_preds['AutoARIMA'] = pd.Series(fc, index=y_test.index)
                except Exception:
                    pass

            # ---------- ETS (candidates) ----------
            try:
                best_fc = None; best_err = np.inf
                for cand in [{'trend':'add','seasonal':'add'},
                             {'trend':'add','seasonal':None},
                             {'trend':None,'seasonal':'add'}]:
                    try:
                        m_ets = ExponentialSmoothing(y_train, trend=cand['trend'], seasonal=cand['seasonal'],
                                                     seasonal_periods=ETS_SEASONAL_PERIODS).fit()
                        fc = m_ets.forecast(len(y_test))
                        s = rmse(y_test, fc)
                        if s < best_err:
                            best_err = s; best_fc = fc
                    except Exception:
                        continue
                if best_fc is not None:
                    model_scores['ETS']['rmse'].append(rmse(y_test, best_fc))
                    model_scores['ETS']['mae'].append(mae(y_test, best_fc))
                    model_scores['ETS']['mape'].append(mape(y_test, best_fc))
                    last_preds['ETS'] = pd.Series(best_fc, index=y_test.index)
            except Exception:
                pass

            # ---------- Prophet ----------
            if HAS_PROPHET:
                try:
                    best_fc = None; best_err = np.inf
                    for cps, sps in [(0.01,0.1),(0.05,0.5),(0.1,1.0)]:
                        try:
                            dfp = y_train.reset_index().rename(columns={'Arrival_Date':'ds','Modal_Price':'y'})
                            m_prop = Prophet(changepoint_prior_scale=cps, seasonality_prior_scale=sps)
                            m_prop.fit(dfp)
                            fut = pd.DataFrame({'ds': y_test.index})
                            fc = m_prop.predict(fut)['yhat'].values
                            s = rmse(y_test, fc)
                            if s < best_err:
                                best_err = s; best_fc = fc
                        except Exception:
                            continue
                    if best_fc is not None:
                        model_scores['Prophet']['rmse'].append(rmse(y_test, best_fc))
                        model_scores['Prophet']['mae'].append(mae(y_test, best_fc))
                        model_scores['Prophet']['mape'].append(mape(y_test, best_fc))
                        last_preds['Prophet'] = pd.Series(best_fc, index=y_test.index)
                except Exception:
                    pass

            # ---------- ML features ----------
            X_train_feat, y_train_ml = make_lag_features(y_train, exog_train, LAGS)
            if X_train_feat.empty:
                continue
            feature_cols = list(X_train_feat.columns)

            # ---- XGBoost ----
            if HAS_XGB:
                try:
                    best_params = optuna_tune("XGBoost", X_train_feat, y_train_ml, n_trials=OPTUNA_TRIALS)
                    if best_params:
                        model_xgb = XGBRegressor(**best_params, random_state=RANDOM_STATE, verbosity=0, n_jobs=-1)
                    else:
                        model_xgb = XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1)
                    model_xgb.fit(X_train_feat, y_train_ml)
                    fc_series = iterative_forecast_ml(model_xgb, y_train, exog_train, exog_test,
                                                      len(y_test), LAGS, feature_cols)
                    model_scores['XGBoost']['rmse'].append(rmse(y_test, fc_series))
                    model_scores['XGBoost']['mae'].append(mae(y_test, fc_series))
                    model_scores['XGBoost']['mape'].append(mape(y_test, fc_series))
                    last_preds['XGBoost'] = fc_series
                except Exception:
                    pass

            # ---- RandomForest ----
            try:
                best_params = optuna_tune("RandomForest", X_train_feat, y_train_ml, n_trials=OPTUNA_TRIALS)
                if best_params:
                    rf = RandomForestRegressor(**best_params, random_state=RANDOM_STATE, n_jobs=-1)
                else:
                    rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
                rf.fit(X_train_feat, y_train_ml)
                fc_series = iterative_forecast_ml(rf, y_train, exog_train, exog_test,
                                                  len(y_test), LAGS, feature_cols)
                model_scores['RandomForest']['rmse'].append(rmse(y_test, fc_series))
                model_scores['RandomForest']['mae'].append(mae(y_test, fc_series))
                model_scores['RandomForest']['mape'].append(mape(y_test, fc_series))
                last_preds['RandomForest'] = fc_series
            except Exception:
                pass

            # ---- SVR ----
            try:
                best_params = optuna_tune("SVR", X_train_feat, y_train_ml, n_trials=OPTUNA_TRIALS)
                scaler = StandardScaler().fit(X_train_feat)
                if best_params:
                    svr = SVR(**best_params)
                else:
                    svr = SVR()
                svr.fit(scaler.transform(X_train_feat), y_train_ml)
                fc_series = iterative_forecast_ml(svr, y_train, exog_train, exog_test,
                                                  len(y_test), LAGS, feature_cols, scaler=scaler)
                model_scores['SVR']['rmse'].append(rmse(y_test, fc_series))
                model_scores['SVR']['mae'].append(mae(y_test, fc_series))
                model_scores['SVR']['mape'].append(mape(y_test, fc_series))
                last_preds['SVR'] = fc_series
            except Exception:
                pass

        # Aggregate CV results
        comp_rows = []
        for mname, met in model_scores.items():
            if met['rmse']:
                avg_rmse = float(np.nanmean(met['rmse']))
                avg_mae  = float(np.nanmean(met['mae']))
                avg_mape = float(np.nanmean(met['mape']))
                best_metric = select_best_metric({'rmse':avg_rmse,'mae':avg_mae,'mape':avg_mape})
                comp_rows.append({
                    "Market": market,
                    "Commodity": commodity,
                    "Model": mname,
                    "Avg_RMSE": avg_rmse,
                    "Avg_MAE": avg_mae,
                    "Avg_MAPE": avg_mape,
                    "Best_Metric": best_metric
                })

        master_partial = pd.DataFrame(comp_rows)
        if not master_partial.empty:
            mfn = os.path.join(PARTIAL_DIR, f"master_partial_{uid}.csv")
            master_partial.to_csv(mfn, index=False)
            # pick best
            chosen_metric = master_partial.iloc[0]["Best_Metric"]
            if chosen_metric == "MAE":
                best_row = master_partial.sort_values("Avg_MAE").iloc[[0]]
            elif chosen_metric == "MAPE":
                best_row = master_partial.sort_values("Avg_MAPE").iloc[[0]]
            elif chosen_metric == "Balanced":
                best_row = master_partial.sort_values("Avg_RMSE").iloc[[0]]
            else:
                best_row = master_partial.sort_values("Avg_RMSE").iloc[[0]]
            bfn = os.path.join(PARTIAL_DIR, f"best_partial_{uid}.csv")
            best_row.to_csv(bfn, index=False)

        return {"status":"ok","market":market,"commodity":commodity}
    except Exception as e:
        return {"status":"error","market":market,"commodity":commodity,"error":str(e)}

def future_forecasting(df_full, best_df, horizon=30, alpha=0.05, n_boot=200):
    forecasts = []
    done_pairs = set()
    if os.path.exists(forecast_ckpt):
        try:
            existing = pd.read_csv(forecast_ckpt)
            done_pairs = set(zip(existing['Market'], existing['Commodity']))
            forecasts.append(existing)
            print(f"Resuming forecast: {len(done_pairs)} pairs already forecasted.")
        except Exception:
            pass

    for _, row in best_df.iterrows():
        market, commodity = row["Market"], row["Commodity"]
        best_model = row.get("Model") or row.get("Best_Model")
        if (market, commodity) in done_pairs:
            continue

        sub_raw = df_full[(df_full['Market'] == market) & (df_full['Commodity'] == commodity)].copy()
        sub_raw['Arrival_Date'] = pd.to_datetime(sub_raw['Arrival_Date'])
        sub_agg = sub_raw.groupby('Arrival_Date').agg({
            'Modal_Price': 'median',
            'Rainfall_mm': 'mean',
            'Avg_Temp_C': 'mean'
        }).asfreq('D').interpolate()

        y = sub_agg['Modal_Price'].dropna()
        if y.empty:
            continue
        exog = sub_agg[['Rainfall_mm','Avg_Temp_C']].loc[y.index] if 'Rainfall_mm' in sub_agg.columns else None
        future_exog = pd.DataFrame(index=pd.date_range(start=y.index[-1] + pd.Timedelta(days=1),
                                                       periods=horizon, freq='D'))
        if exog is not None:
            for c in exog.columns:
                future_exog[c] = exog[c].iloc[-7:].mean()

        fc, lower, upper, fc_index = None, None, None, None

        try:
            if best_model == "AutoARIMA" and HAS_PMD:
                m_ar = pm.auto_arima(y, exogenous=exog, seasonal=True, m=7,
                                     stepwise=True, suppress_warnings=True)
                fc, conf = m_ar.predict(n_periods=horizon, exogenous=future_exog,
                                        return_conf_int=True, alpha=alpha)
                lower, upper = conf[:,0], conf[:,1]
                fc_index = future_exog.index
            elif best_model == "ETS":
                y_stationary = ensure_stationary(y)
                m_ets = ExponentialSmoothing(y_stationary, trend="add", seasonal="add",
                                             seasonal_periods=ETS_SEASONAL_PERIODS).fit()
                fc_series = m_ets.forecast(horizon)
                fc, fc_index = fc_series.values, fc_series.index
                fitted = m_ets.fittedvalues
                residuals = (y - fitted).dropna().values
                lower, upper = bootstrap_intervals(fc, residuals, horizon, n_boot, alpha)
            elif best_model == "Prophet" and HAS_PROPHET:
                dfp = y.reset_index().rename(columns={"Arrival_Date":"ds","Modal_Price":"y"})
                mp = Prophet(interval_width=1-alpha).fit(dfp)
                fut = pd.DataFrame({"ds": future_exog.index})
                pred = mp.predict(fut)
                fc, lower, upper = pred["yhat"].values, pred["yhat_lower"].values, pred["yhat_upper"].values
                fc_index = pd.DatetimeIndex(pred["ds"].values)
            else:
                # load saved artifact if exists
                art_path = _find_artifact_for_pair(market, commodity, best_model)
                model_obj = None; scaler = None; feature_cols = None
                if art_path and os.path.exists(art_path):
                    art = joblib.load(art_path)
                    if isinstance(art, dict):
                        model_obj = art.get('model')
                        scaler = art.get('scaler')
                        feature_cols = art.get('feature_cols')
                X_feat, y_ml = make_lag_features(y, exog, LAGS)
                if feature_cols is None:
                    feature_cols = list(X_feat.columns)
                if model_obj is None:
                    if best_model == 'XGBoost' and HAS_XGB:
                        model_obj = XGBRegressor(random_state=RANDOM_STATE).fit(X_feat, y_ml)
                    elif best_model == 'RandomForest':
                        model_obj = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE).fit(X_feat, y_ml)
                    elif best_model == 'SVR':
                        scaler = StandardScaler().fit(X_feat)
                        model_obj = SVR().fit(scaler.transform(X_feat), y_ml)
                if model_obj is not None:
                    fc_series = iterative_forecast_ml(model_obj, y, exog, future_exog,
                                                      horizon, LAGS, feature_cols, scaler=scaler)
                    fc, fc_index = fc_series.values, fc_series.index
                    fitted = model_obj.predict(scaler.transform(X_feat) if scaler else X_feat)
                    residuals = y_ml.values - fitted
                    lower, upper = bootstrap_intervals(fc, residuals, horizon, n_boot, alpha)

            if fc is not None and fc_index is not None:
                tmp = pd.DataFrame({
                    "Date": fc_index,
                    "Market": market,
                    "Commodity": commodity,
                    "Forecast": fc,
                    "Lower_CI": list(lower) if lower is not None else [np.nan]*len(fc),
                    "Upper_CI": list(upper) if upper is not None else [np.nan]*len(fc),
                    "Model": best_model
                })
                forecasts.append(tmp)
                pd.concat(forecasts, ignore_index=True).to_csv(forecast_ckpt, index=False)
        except Exception as e:
            print(f"Forecast failed for {market}-{commodity}: {e}")

    if forecasts:
        out = pd.concat(forecasts, ignore_index=True)
        out.to_csv(os.path.join(OUTPUT_DIR, "future_forecast.csv"), index=False)
        return out
    else:
        return pd.DataFrame()

In [ ]:
def main_parallel_full(csv_path=CSV_PATH, n_workers=N_WORKERS, auto_merge_every=AUTO_MERGE_EVERY):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found at {csv_path}")
    df = pd.read_csv(csv_path)
    df = clean_full_df(df)

    pairs_df = df.groupby(['Market','Commodity']).size().reset_index(name='count')
    pairs = pairs_df[['Market','Commodity']].to_records(index=False).tolist()
    total_pairs = len(pairs)

    done_pairs = set()
    if os.path.exists(BEST_FINAL):
        try:
            best_done = pd.read_csv(BEST_FINAL)
            done_pairs = set(zip(best_done['Market'], best_done['Commodity']))
            print(f"Resuming run: {len(done_pairs)} pairs already completed.")
        except Exception:
            pass

    pairs_to_run = [(m, c, df) for (m, c) in pairs if (m, c) not in done_pairs]
    print(f"Total pairs: {total_pairs} | Remaining: {len(pairs_to_run)} | Workers: {n_workers}")

    start_time = time.time()
    completed = 0
    failed = 0

    if pairs_to_run:
        with mp.Pool(processes=n_workers) as pool:
            for res in pool.imap_unordered(run_pair_worker, pairs_to_run):
                completed += 1
                elapsed = time.time() - start_time
                avg_time = elapsed / completed if completed > 0 else 0
                remaining = len(pairs_to_run) - completed
                eta_minutes = (remaining * avg_time) / 60.0
                if isinstance(res, dict) and res.get("status") == "error":
                    failed += 1
                print(f"✅ Completed {completed}/{len(pairs_to_run)} this run | ETA: {eta_minutes:.1f} min | Failed: {failed}")
                if auto_merge_every and completed % auto_merge_every == 0:
                    merge_partials(PARTIAL_DIR, MASTER_FINAL, BEST_FINAL)

    master_all, best_all = save_artifact(PARTIAL_DIR, MASTER_FINAL, BEST_FINAL)
    if not best_all.empty:
        future_forecasting(df, best_all, horizon=HORIZON, alpha=ALPHA, n_boot=BOOTSTRAP_N)
    return master_all, best_all